# Model 1  XLM-RoBERTa Song Ngữ (Anh + Việt)
Fine-tune `xlm-roberta-base` trên cả tiếng Anh lẫn tiếng Việt cùng lúc.
- Input: `processed_data/train.csv`, `processed_data/val.csv` (toàn bộ)
- Output: `models/xlmroberta/`

In [1]:
!pip install transformers scikit-learn torch sentencepiece -q

In [ ]:
MODEL_NAME = "xlm-roberta-base"
OUTPUT_DIR = "models/xlmroberta"
MAX_LEN    = 128
BATCH_SIZE = 32   # phải giảm batch size khi tăng MAX_LEN, tránh OOM
EPOCHS     = 3
LR         = 2e-5
SEED       = 42

In [3]:
import pandas as pd, torch
from pathlib import Path
torch.manual_seed(SEED)

# Không filter ngôn ngữ  dùng toàn bộ
train_df = pd.read_csv("/kaggle/input/datasets/minhbodoi/faidset-processed/train.csv", encoding="utf-8-sig")
val_df   = pd.read_csv("/kaggle/input/datasets/minhbodoi/faidset-processed/val.csv",   encoding="utf-8-sig")

print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}")
print(f"\nPhân phối train:")
print(train_df.groupby(["language", "label"]).size().to_string())

Train: 44,283  |  Val: 4,921

Phân phối train:
language  label
en        0        12237
          1        14989
vi        0        10000
          1         7057


In [4]:
from torch.utils.data import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.texts   = df["text"].tolist()
        self.labels  = df["label"].tolist()
        self.tok     = tokenizer
        self.max_len = max_len
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=self.max_len,
                       padding="max_length", truncation=True, return_tensors="pt")
        return {"input_ids":      enc["input_ids"].squeeze(),
                "attention_mask": enc["attention_mask"].squeeze(),
                "label":          torch.tensor(self.labels[idx], dtype=torch.long)}

train_dataset = TextDataset(train_df, tokenizer, MAX_LEN)
val_dataset   = TextDataset(val_df,   tokenizer, MAX_LEN)
print(f" Dataset ready  train: {len(train_dataset):,}, val: {len(val_dataset):,}")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

 Dataset ready  train: 44,283, val: 4,921


In [5]:
from transformers import AutoModelForSequenceClassification
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

Device: cuda


model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [6]:
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
optimizer    = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * EPOCHS
scheduler    = get_linear_schedule_with_warmup(optimizer, int(0.1*total_steps), total_steps)

# Class weight tính trên EN: AI nhiều hơn human → tăng penalty cho human
# VI đã F1=0.99 nên không cần tune
cw_en = compute_class_weight('balanced', classes=np.array([0, 1]),
                              y=train_df[train_df["language"] == "en"]["label"].values)
cw_tensor = torch.tensor(cw_en, dtype=torch.float).to(device)
loss_fn = torch.nn.CrossEntropyLoss(weight=cw_tensor)
print(f"Class weights EN: human={cw_en[0]:.4f}, AI={cw_en[1]:.4f}")

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
best_f1 = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"].to(device),
                    attention_mask=batch["attention_mask"].to(device))
        loss = loss_fn(out.logits, batch["label"].to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        total_loss += loss.item()

    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(input_ids=batch["input_ids"].to(device),
                        attention_mask=batch["attention_mask"].to(device))
            preds.extend(out.logits.argmax(-1).cpu().numpy())
            trues.extend(batch["label"].numpy())

    val_f1 = f1_score(trues, preds, average="macro")
    print(f"Epoch {epoch+1}/{EPOCHS} | loss: {total_loss/len(train_loader):.4f} | val F1: {val_f1:.4f}")
    if val_f1 > best_f1:
        best_f1 = val_f1
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        print(f"   Saved (F1={best_f1:.4f})")

print(f"\n Done. Best val F1: {best_f1:.4f}")


Class weights EN: human=1.1124, AI=0.9082
Epoch 1/3 | loss: 0.2870 | val F1: 0.8837


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Saved (F1=0.8837)
Epoch 2/3 | loss: 0.1291 | val F1: 0.8798
Epoch 3/3 | loss: 0.0788 | val F1: 0.9004


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

   Saved (F1=0.9004)

 Done. Best val F1: 0.9004


In [7]:
# Đánh giá test set  tổng và theo từng ngôn ngữ
from transformers import AutoModelForSequenceClassification, AutoTokenizer

test_en_df = pd.read_csv("/kaggle/input/datasets/minhbodoi/faidset-processed/test_en.csv", encoding="utf-8-sig")
test_vi_df = pd.read_csv("/kaggle/input/datasets/minhbodoi/faidset-processed/test_vi.csv", encoding="utf-8-sig")
test_df    = pd.concat([test_en_df, test_vi_df]).reset_index(drop=True)
model_best = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR).to(device)
tok_best   = AutoTokenizer.from_pretrained(OUTPUT_DIR)
test_dataset = TextDataset(test_df, tok_best, MAX_LEN)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

model_best.eval()
preds, trues = [], []
with torch.no_grad():
    for batch in test_loader:
        out = model_best(input_ids=batch["input_ids"].to(device),
                         attention_mask=batch["attention_mask"].to(device))
        preds.extend(out.logits.argmax(-1).cpu().numpy())
        trues.extend(batch["label"].numpy())

test_df["pred"] = preds
print(" XLM-RoBERTa  Tổng:")
print(classification_report(trues, preds, target_names=["Human", "AI"]))

for lang in ["en", "vi"]:
    sub = test_df[test_df["language"] == lang]
    print(f"\n {lang.upper()} ")
    print(classification_report(sub["label"], sub["pred"], target_names=["Human", "AI"]))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

 XLM-RoBERTa  Tổng:
              precision    recall  f1-score   support

       Human       0.93      0.53      0.67     62683
          AI       0.67      0.96      0.79     63496

    accuracy                           0.75    126179
   macro avg       0.80      0.74      0.73    126179
weighted avg       0.80      0.75      0.73    126179


 EN 
              precision    recall  f1-score   support

       Human       0.92      0.52      0.66     60724
          AI       0.67      0.96      0.79     62176

    accuracy                           0.74    122900
   macro avg       0.80      0.74      0.73    122900
weighted avg       0.80      0.74      0.73    122900


 VI 
              precision    recall  f1-score   support

       Human       1.00      0.97      0.99      1959
          AI       0.96      1.00      0.98      1320

    accuracy                           0.98      3279
   macro avg       0.98      0.99      0.98      3279
weighted avg       0.98      0.98      0.9

In [8]:
# Tự động upload model lên Kaggle Dataset sau khi train xong
import json, os, subprocess

dataset_name = "xlmroberta-model"
kaggle_user  = subprocess.run('kaggle config view', shell=True,
                              capture_output=True, text=True).stdout
kaggle_user  = [l.split(':')[1].strip() for l in kaggle_user.split('\n')
                if 'username' in l][0]

upload_dir = "/kaggle/working/models/xlmroberta"
os.makedirs(upload_dir, exist_ok=True)

with open(f'/kaggle/working/models/xlmroberta/dataset-metadata.json', 'w') as f:
    json.dump({
        'title'   : dataset_name,
        'id'      : f'{kaggle_user}/{dataset_name}',
        'licenses': [{'name': 'CC0-1.0'}]
    }, f)

check = subprocess.run(f'kaggle datasets list --user {kaggle_user} --search {dataset_name}',
                       shell=True, capture_output=True, text=True)
if dataset_name in check.stdout:
    result = subprocess.run(f'kaggle datasets version -p {upload_dir} -m "auto update"',
                            shell=True, capture_output=True, text=True)
else:
    result = subprocess.run(f'kaggle datasets create -p {upload_dir}',
                            shell=True, capture_output=True, text=True)

print(result.stdout)
print(result.stderr)
print(f'Done! {kaggle_user}/{dataset_name}')


Starting upload for file model.safetensors
Upload successful: model.safetensors (1GB)
Starting upload for file config.json
Upload successful: config.json (761B)
Starting upload for file tokenizer.json
Upload successful: tokenizer.json (16MB)
Starting upload for file tokenizer_config.json
Upload successful: tokenizer_config.json (314B)
Dataset version is being created. Please check progress at https://www.kaggle.com/datasets/minhbodoi/xlmroberta-model


  0%|          | 0.00/1.04G [00:00<?, ?B/s]
  0%|          | 3.98M/1.04G [00:00<00:26, 41.7MB/s]
  1%|▏         | 13.8M/1.04G [00:00<00:14, 76.9MB/s]
  2%|▏         | 21.1M/1.04G [00:00<00:17, 62.9MB/s]
  3%|▎         | 27.4M/1.04G [00:00<00:17, 63.6MB/s]
  3%|▎         | 36.2M/1.04G [00:00<00:16, 67.1MB/s]
  4%|▍         | 42.7M/1.04G [00:00<00:21, 50.8MB/s]
  5%|▍         | 48.0M/1.04G [00:00<00:23, 45.0MB/s]
  5%|▌         | 54.4M/1.04G [00:01<00:21, 49.7MB/s]
  6%|▌         | 59.5M/1.04G [00:01<00:24, 42.8MB/s]
  6%|▌         | 64.5M